# Tensor RL — All Cases vs Baseline on MiniGrid

Runs all 6 tensor decomposition cases against a standard DQN baseline on `MiniGrid-Empty-5x5-v0`.

| Case | Description | Key flags |
|---|---|---|
| Baseline | Standard DQN | `--network standard` |
| 1 | CP tensor network | `--network cp` |
| 2a | Pre-init Tucker | `--network tucker` |
| 2b | Post-hoc CP compression | `--compress cp` |
| 3 | Structured TT embedding | `--structured --network tt` |
| 4a | CNN backbone + CP head | `--cnn_mode backbone --network cp` |
| 4b | Tensorized CNN (Tucker conv) | `--cnn_mode tensorized --network tucker` |
| 6 | MPS (true TT linear layers) | `--network mps` |

### Why observations are normalised
MiniGrid `ImgObsWrapper` returns integer values in `[0, 10]` (object type / colour / door state).
Without scaling, Q-network inputs are an order of magnitude larger than the weights expect from Kaiming
init, causing Q-values to saturate and gradients to vanish. `env_utils.make_env` now divides by 10,
mapping observations to `[0, 1]`. This is the single biggest factor for stable learning.

### Why 500 episodes?
With `epsilon_decay_steps = N_EPISODES × 40` and ~50 steps/episode on a solved trajectory,
epsilon reaches `epsilon_end = 0.05` after roughly 40 episodes. The remaining ~460 episodes are
exploitation, giving the Q-function time to consolidate. 200 episodes left epsilon at ~0.3 — still
mostly exploring, so the greedy (eval) policy looked untrained.

In [5]:
import sys, os
sys.path.insert(0, 'sim')
print(os.getcwd())
os.chdir('sim')  # train.py expects to run from sim/

# ── CONFIG ──────────────────────────────────────────────────────────────────
ENV        = 'MiniGrid-Empty-5x5-v0'
N_EPISODES = 500     # MiniGrid needs ~400+ eps; 200 is too short (epsilon still 0.3 at end)
RANK       = 4       # tensor rank / bond dimension
SEED       = 42
# ────────────────────────────────────────────────────────────────────────────

# Epsilon decay is set to episodes * 40 in train.py, meaning epsilon reaches
# epsilon_end (~0.05) after 40 * avg_steps_per_episode ≈ 40*50 = 2000 steps,
# i.e. after ~40 episodes. The agent then exploits for the remaining ~460 episodes.

print(f'env={ENV}  episodes={N_EPISODES}  rank={RANK}  seed={SEED}')

/home/octyo/github/tensor_rl/sim/sim


FileNotFoundError: [Errno 2] No such file or directory: 'sim'

In [ ]:
import argparse
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

from envs.env_utils import make_env, make_env_structured, make_env_cnn
from train import run_one_seed, run_with_compression, build_run_name
from analysis.stats import smooth

# double_dqn uses Polyak soft target updates (tau) + SmoothL1 loss + Double-DQN action selection.
# It is far more stable than vanilla DQN on MiniGrid where solved episodes are very short
# (~10 steps), causing hard target copies every 1000 steps to be extremely stale.
ALGO = 'double_dqn'

def make_args(**overrides):
    """Build a minimal args namespace for train.py functions."""
    defaults = dict(
        env=ENV, algo=ALGO, network='standard',
        rank=RANK, episodes=N_EPISODES,
        tensorize_layers='all', structured=False,
        cnn_mode=None, compress=None, finetune_episodes=0,
        tt_dims=None, tau=0.05, seeds=str(SEED),
        max_steps=200, wandb=False,
    )
    defaults.update(overrides)
    return argparse.Namespace(**defaults)

results = {}   # label → metrics dict
print(f'Setup complete. algo={ALGO}')

## Run all cases
Each cell runs one case and stores results. Re-run individual cells to rerun a single case.

In [4]:
# ── BASELINE: Standard DQN ───────────────────────────────────────────────────
label = 'Baseline (standard)'
args = make_args(network='standard')
env  = make_env(ENV)
metrics, _ = run_one_seed(env, args, SEED, build_run_name(args, SEED))
results[label] = metrics
print(f'{label}: final eval reward = {np.mean(metrics["eval_reward"][-20:]):.3f}')

  [standard/layers:all] 36359 params (~142.03 KB)
  Ep 50/500 | train=0.00 | eval=0.00 | loss=0.0105 | eps=0.83
  Ep 100/500 | train=0.69 | eval=0.00 | loss=0.0035 | eps=0.66
  Ep 150/500 | train=0.00 | eval=0.00 | loss=0.0041 | eps=0.46
  Ep 200/500 | train=0.00 | eval=0.00 | loss=0.0051 | eps=0.23
  Ep 250/500 | train=0.00 | eval=0.00 | loss=0.0062 | eps=0.05
  Ep 300/500 | train=0.00 | eval=0.00 | loss=0.0047 | eps=0.05
  Ep 350/500 | train=0.00 | eval=0.00 | loss=0.0040 | eps=0.05
  Ep 400/500 | train=0.00 | eval=0.00 | loss=0.0044 | eps=0.05
  Ep 450/500 | train=0.00 | eval=0.00 | loss=0.0042 | eps=0.05
  Ep 500/500 | train=0.00 | eval=0.00 | loss=0.0039 | eps=0.05
Baseline (standard): final eval reward = 0.000


In [5]:
# ── CASE 1: CP tensor network ────────────────────────────────────────────────
label = 'Case 1: CP layers'
args = make_args(network='cp')
env  = make_env(ENV)
metrics, _ = run_one_seed(env, args, SEED, build_run_name(args, SEED))
results[label] = metrics
print(f'{label}: params={metrics.get("params", "?")}  final eval={np.mean(metrics["eval_reward"][-20:]):.3f}')

  [cp/layers:all] 2927 params (~11.43 KB)
  Ep 50/200 | train=0.24 | eval=0.00 | loss=0.0059 | eps=0.83
  Ep 100/200 | train=0.00 | eval=0.00 | loss=0.0025 | eps=0.68
  Ep 150/200 | train=0.87 | eval=0.00 | loss=0.0024 | eps=0.54
  Ep 200/200 | train=0.00 | eval=0.00 | loss=0.0029 | eps=0.36
Case 1: CP layers: params=2927  final eval=0.000


In [ ]:
# ── CASE 2a: Pre-init Tucker layers ─────────────────────────────────────────
label = 'Case 2a: Tucker layers'
args = make_args(network='tucker')
env  = make_env(ENV)
metrics, _ = run_one_seed(env, args, SEED, build_run_name(args, SEED))
results[label] = metrics
print(f'{label}: params={metrics.get("params", "?")}  final eval={np.mean(metrics["eval_reward"][-20:]):.3f}')

In [ ]:
# ── CASE 2b: Post-hoc compression (train standard → compress to CP) ──────────
# Note: run_with_compression overrides network='standard' internally for the pre-train phase.
# The algo (double_dqn) is passed through so Polyak updates are used.
label = 'Case 2b: Post-hoc CP compress'
args = make_args(network='standard', compress='cp', finetune_episodes=20)
env  = make_env(ENV)
metrics = run_with_compression(env, args, SEED, build_run_name(args, SEED))
results[label] = metrics
print(f'{label}: params={metrics.get("params", "?")}  final eval={np.mean(metrics["eval_reward"][-20:]):.3f}')

In [ ]:
# ── CASE 3: Structured TT embedding (spatial input preserved) ────────────────
label = 'Case 3: Structured TT'
args = make_args(network='tt', structured=True)
env, mode_dims = make_env_structured(ENV)
print(f'  mode_dims: {mode_dims}')
metrics, _ = run_one_seed(env, args, SEED, build_run_name(args, SEED), mode_dims=mode_dims)
results[label] = metrics
print(f'{label}: params={metrics.get("params", "?")}  final eval={np.mean(metrics["eval_reward"][-20:]):.3f}')

In [ ]:
# ── CASE 4a: CNN backbone + CP linear head ───────────────────────────────────
label = 'Case 4a: CNN backbone + CP head'
args = make_args(network='cp', cnn_mode='backbone')
env, obs_shape = make_env_cnn(ENV)
print(f'  obs_shape (C,H,W): {obs_shape}')
metrics, _ = run_one_seed(env, args, SEED, build_run_name(args, SEED), obs_shape=obs_shape)
results[label] = metrics
print(f'{label}: params={metrics.get("params", "?")}  final eval={np.mean(metrics["eval_reward"][-20:]):.3f}')

In [ ]:
# ── CASE 4b: Tensorized CNN (Tucker conv layers) ─────────────────────────────
label = 'Case 4b: Tensorized CNN (Tucker conv)'
args = make_args(network='tucker', cnn_mode='tensorized')
env, obs_shape = make_env_cnn(ENV)
metrics, _ = run_one_seed(env, args, SEED, build_run_name(args, SEED), obs_shape=obs_shape)
results[label] = metrics
print(f'{label}: params={metrics.get("params", "?")}  final eval={np.mean(metrics["eval_reward"][-20:]):.3f}')

In [ ]:
# ── CASE 5/6: True MPS linear layers ─────────────────────────────────────────
label = 'Case 6: MPS linear'
args = make_args(network='mps')
env  = make_env(ENV)
metrics, _ = run_one_seed(env, args, SEED, build_run_name(args, SEED))
results[label] = metrics
print(f'{label}: params={metrics.get("params", "?")}  final eval={np.mean(metrics["eval_reward"][-20:]):.3f}')

## Results

In [ ]:
# ── Summary table ─────────────────────────────────────────────────────────────
print(f'{'Case':<38} {'Params':>8}  {'Train rew (last20)':>20}  {'Eval rew (last20)':>18}')
print('-' * 90)
for label, m in results.items():
    params     = m.get('params', 0)
    train_last = np.mean(m['rewards'][-20:])      if m.get('rewards')      else float('nan')
    eval_last  = np.mean(m['eval_reward'][-20:])  if m.get('eval_reward')  else float('nan')
    print(f'{label:<38} {params:>8}  {train_last:>20.3f}  {eval_last:>18.3f}')

In [ ]:
# ── Learning curves: eval reward ──────────────────────────────────────────────
SMOOTH = max(1, N_EPISODES // 20)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Colour the baseline differently
cmap = plt.get_cmap('tab10')
colours = {label: ('black' if 'Baseline' in label else cmap(i))
           for i, label in enumerate(results)}
lw      = {label: (2.5 if 'Baseline' in label else 1.5) for label in results}
ls      = {label: ('--' if 'Baseline' in label else '-') for label in results}

for label, m in results.items():
    eps  = m.get('episodes', list(range(1, len(m['rewards']) + 1)))
    kw   = dict(label=label, color=colours[label], lw=lw[label], ls=ls[label])
    if m.get('rewards'):
        axes[0].plot(eps, smooth(m['rewards'],     SMOOTH), **kw)
    if m.get('eval_reward'):
        axes[1].plot(eps, smooth(m['eval_reward'], SMOOTH), **kw)

for ax, title, ylabel in [
    (axes[0], 'Train Reward',  'Episode Reward'),
    (axes[1], 'Eval Reward',   'Mean Eval Reward (5 eps)'),
]:
    ax.set_title(title)
    ax.set_xlabel('Episode')
    ax.set_ylabel(ylabel)
    ax.legend(fontsize=7, loc='lower right')
    ax.grid(True, alpha=0.3)

fig.suptitle(f'All Cases vs Baseline — {ENV} | rank={RANK} | {N_EPISODES} episodes', fontsize=11)
plt.tight_layout()
plt.savefig('../figures/learning_curves.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Parameter efficiency scatter ──────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(9, 5))

for label, m in results.items():
    params = m.get('params', None)
    if params is None:
        continue
    eval_final = np.mean(m['eval_reward'][-20:]) if m.get('eval_reward') else float('nan')
    marker = '*' if 'Baseline' in label else 'o'
    ax.scatter(params, eval_final, s=120, marker=marker,
               color=colours[label], zorder=3, label=label)
    ax.annotate(label.split(':')[-1].strip(), (params, eval_final),
                fontsize=7, textcoords='offset points', xytext=(6, 3))

ax.set_xlabel('Parameter Count')
ax.set_ylabel('Mean Eval Reward (last 20 episodes)')
ax.set_title(f'Parameter Efficiency — {ENV}')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('../figures/param_efficiency.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Training stability: grad norm + mean Q ────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

for label, m in results.items():
    eps = m.get('episodes', list(range(1, len(m.get('grad_norm', [])) + 1)))
    kw  = dict(label=label, color=colours[label], lw=lw[label], ls=ls[label], alpha=0.85)
    if m.get('grad_norm'):
        axes[0].plot(eps, smooth(m['grad_norm'], SMOOTH), **kw)
    if m.get('mean_q'):
        axes[1].plot(eps, smooth(m['mean_q'],    SMOOTH), **kw)

axes[0].set_title('Gradient Norm');  axes[0].set_ylabel('Grad Norm')
axes[1].set_title('Mean Max Q');     axes[1].set_ylabel('Mean Q')
for ax in axes:
    ax.set_xlabel('Episode')
    ax.legend(fontsize=7)
    ax.grid(True, alpha=0.3)

fig.suptitle('Training Stability', fontsize=11)
plt.tight_layout()
plt.savefig('../figures/stability.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Compression ratio bar chart ───────────────────────────────────────────────
baseline_params = results.get('Baseline (standard)', {}).get('params', None)

if baseline_params:
    labels_sorted = sorted(
        [(l, m.get('params', baseline_params)) for l, m in results.items() if m.get('params')],
        key=lambda x: x[1]
    )
    names  = [l.split(':')[-1].strip() for l, _ in labels_sorted]
    ratios = [baseline_params / p for _, p in labels_sorted]
    bar_colours = ['black' if 'standard' in n.lower() or 'Baseline' in n else '#4C72B0'
                   for n in names]

    fig, ax = plt.subplots(figsize=(10, 4))
    bars = ax.barh(names, ratios, color=bar_colours, edgecolor='white')
    ax.axvline(1.0, color='red', lw=1.5, ls='--', label='1× (baseline)')
    ax.set_xlabel('Compression ratio (baseline params / variant params)')
    ax.set_title('Parameter Compression vs Baseline')
    for bar, r in zip(bars, ratios):
        ax.text(r + 0.05, bar.get_y() + bar.get_height()/2,
                f'{r:.1f}×', va='center', fontsize=8)
    ax.legend()
    plt.tight_layout()
    plt.savefig('../figures/compression_ratio.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print('Baseline params not recorded — skipping compression chart.')

In [ ]:
# ── Final score table (ranked by eval reward) ─────────────────────────────────
rows = []
bp = results.get('Baseline (standard)', {}).get('params', 1)
for label, m in results.items():
    params    = m.get('params', 0)
    eval_mean = np.mean(m['eval_reward'][-20:]) if m.get('eval_reward') else float('nan')
    ratio     = bp / params if params else float('nan')
    rows.append((label, params, ratio, eval_mean))

rows.sort(key=lambda x: -x[3])  # sort by eval reward descending
print(f'{'Case':<38} {'Params':>8}  {'Compress':>10}  {'Eval Reward':>12}')
print('-' * 75)
for label, params, ratio, ev in rows:
    ratio_s = f'{ratio:.1f}×' if ratio == ratio else 'n/a'
    print(f'{label:<38} {params:>8}  {ratio_s:>10}  {ev:>12.4f}')